# More Trees

## Classification Trees

* Implementing the Classification Tree Class from scratch using only `NumPy`. The splitting criterion is Gini impurity.
* Running the implementation on the `IRIS` classification dataset.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris

class DecisionTree:
    """Base class for a decision tree."""

    def fit(self, X, y):
        """Fit the decision tree to the training data."""
        raise NotImplementedError("fit() must be implemented in subclass.")

    def predict(self, X):
        """Predict class labels for given samples."""
        raise NotImplementedError("predict() must be implemented in subclass.")


In [ ]:
class ClassificationTree(DecisionTree):
    """Classification decision tree using Gini impurity."""

    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.tree = None

    def gini(self, y):
        """Compute Gini impurity for labels y."""
        classes, counts = np.unique(y, return_counts=True)
        p = counts / len(y)
        return 1.0 - np.sum(p ** 2)

    def best_split(self, X, y):
        """Return best feature and threshold for splitting based on Gini."""
        best_feature, best_threshold, best_gini = None, None, np.inf
        n_samples, n_features = X.shape

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])
            for thr in thresholds:
                left = X[:, feature] <= thr
                right = ~left

                # skip if split fails
                if np.sum(left) == 0 or np.sum(right) == 0:
                    continue

                g_left = self.gini(y[left])
                g_right = self.gini(y[right])
                g_split = (len(y[left]) * g_left + len(y[right]) * g_right) / len(y)

                if g_split < best_gini:
                    best_gini = g_split
                    best_feature = feature
                    best_threshold = thr

        return best_feature, best_threshold

    def build(self, X, y, depth=0):
        """Recursive tree builder."""
        classes, counts = np.unique(y, return_counts=True)
        prediction = classes[np.argmax(counts)]

        # stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
           len(np.unique(y)) == 1 or len(y) < self.min_samples_split:
            return {"leaf": True, "prediction": prediction}

        feature, threshold = self.best_split(X, y)
        if feature is None:
            return {"leaf": True, "prediction": prediction}

        left = X[:, feature] <= threshold
        right = ~left

        return {
            "leaf": False,
            "feature": feature,
            "threshold": threshold,
            "left": self.build(X[left], y[left], depth + 1),
            "right": self.build(X[right], y[right], depth + 1),
        }

    def fit(self, X, y):
        """Train the decision tree."""
        X = np.array(X)
        y = np.array(y)
        self.tree = self.build(X, y)
        return self

    def _predict_one(self, x, node):
        """Predict for a single sample."""
        if node["leaf"]:
            return node["prediction"]
        if x[node["feature"]] <= node["threshold"]:
            return self._predict_one(x, node["left"])
        else:
            return self._predict_one(x, node["right"])

    def predict(self, X):
        """Predict for multiple samples."""
        X = np.array(X)
        return np.array([self._predict_one(x, self.tree) for x in X])


In [ ]:
#tree classifier on iris dataset
iris = load_iris()
X, y = iris.data, iris.target

np.random.seed(42)
indices = np.random.permutation(len(X))
train_size = int(0.7 * len(X))
train_idx = indices[:train_size]
test_idx = indices[train_size:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

tree = ClassificationTree(max_depth=5, min_samples_split=2)
tree.fit(X_train, y_train)
y_pred = tree.predict(X_test)

accuracy = np.mean(y_pred == y_test)

print("(RESULT) Accuracy on IRIS using custom ClassificationTree:", accuracy)


(RESULT) Accuracy on IRIS using custom ClassificationTree: 0.9555555555555556


## Random Forests

* Implementing Random Forests using only `NumPy`.
* Comparing the results between the random forest run of the `ClassificationTree` class on the `IRIS` dataset.

In [ ]:
class RandomForest:
    """Random Forest Classifier using only NumPy and the custom ClassificationTree."""

    def __init__(self, n_trees=10, max_depth=None, min_samples_split=2, sample_ratio=0.8, feature_ratio=0.8):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.sample_ratio = sample_ratio
        self.feature_ratio = feature_ratio
        self.trees = []
        self.feature_subsets = []

    def bootstrap_sample(self, X, y):
        n_samples = len(X)
        idx = np.random.choice(n_samples, int(n_samples * self.sample_ratio), replace=True)
        return X[idx], y[idx]

    def feature_subset(self, X):
        n_features = X.shape[1]
        m = max(1, int(n_features * self.feature_ratio))
        idx = np.random.choice(n_features, m, replace=False)
        return idx

    def fit(self, X, y):
        self.trees = []
        self.feature_subsets = []

        for _ in range(self.n_trees):
            X_boot, y_boot = self.bootstrap_sample(X, y)
            f_idx = self.feature_subset(X_boot)

            tree = ClassificationTree(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            tree.fit(X_boot[:, f_idx], y_boot)

            self.trees.append(tree)
            self.feature_subsets.append(f_idx)

        return self

    def predict(self, X):
        predictions = []
        for tree, f_idx in zip(self.trees, self.feature_subsets):
            preds = tree.predict(X[:, f_idx])
            predictions.append(preds)

        preds = np.array(predictions)      # shape: (n_trees, n_samples)
        final_preds = []
        for sample in preds.T:             # majority vote for each sample
            values, counts = np.unique(sample, return_counts=True)
            final_preds.append(values[np.argmax(counts)])

        return np.array(final_preds)


In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

np.random.seed(42)
indices = np.random.permutation(len(X))
train_size = int(0.7 * len(X))
train_idx = indices[:train_size]
test_idx = indices[train_size:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]


# Single tree
single_tree = ClassificationTree(max_depth=5)
single_tree.fit(X_train, y_train)
y_pred_tree = single_tree.predict(X_test)
acc_tree = np.mean(y_pred_tree == y_test)

# Random forest
forest = RandomForest(n_trees=10, max_depth=5)
forest.fit(X_train, y_train)
y_pred_forest = forest.predict(X_test)
acc_forest = np.mean(y_pred_forest == y_test)

print("(RESULT) Accuracy of ClassificationTree:", acc_tree)
print("(RESULT) Accuracy of RandomForest:", acc_forest)


(RESULT) Accuracy of ClassificationTree: 0.9555555555555556
(RESULT) Accuracy of RandomForest: 0.9555555555555556


## Extra Trees

* Implement Extra Trees using only `NumPy`.
* Comparing the results between the `Random Forest` and an `Extra Trees` ensemble implementation on the `IRIS` dataset.

In [ ]:
class ExtraTree(DecisionTree):
    """Extremely Randomized Tree - uses random thresholds instead of optimal ones."""

    def __init__(self, max_depth=None, min_samples_split=2, n_thresholds=10):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_thresholds = n_thresholds
        self.tree = None

    def gini(self, y):
        classes, counts = np.unique(y, return_counts=True)
        p = counts / len(y)
        return 1 - np.sum(p ** 2)

    def random_split(self, X, y):
        n_samples, n_features = X.shape
        best_feature, best_thresh, best_gini = None, None, np.inf

        for feature in range(n_features):
            values = X[:, feature]
            min_val, max_val = np.min(values), np.max(values)
            thresholds = np.random.uniform(min_val, max_val, self.n_thresholds)

            for thr in thresholds:
                left = X[:, feature] <= thr
                right = ~left
                if np.sum(left) == 0 or np.sum(right) == 0:
                    continue

                g_left = self.gini(y[left])
                g_right = self.gini(y[right])
                g = (len(y[left]) * g_left + len(y[right]) * g_right) / len(y)

                if g < best_gini:
                    best_gini = g
                    best_feature = feature
                    best_thresh = thr

        return best_feature, best_thresh

    def build(self, X, y, depth=0):
        classes, counts = np.unique(y, return_counts=True)
        prediction = classes[np.argmax(counts)]

        if (self.max_depth is not None and depth >= self.max_depth) or \
           len(np.unique(y)) == 1 or len(y) < self.min_samples_split:
            return {"leaf": True, "prediction": prediction}

        feature, threshold = self.random_split(X, y)
        if feature is None:
            return {"leaf": True, "prediction": prediction}

        left = X[:, feature] <= threshold
        right = ~left

        return {
            "leaf": False,
            "feature": feature,
            "threshold": threshold,
            "left": self.build(X[left], y[left], depth + 1),
            "right": self.build(X[right], y[right], depth + 1)
        }

    def fit(self, X, y):
        X, y = np.array(X), np.array(y)
        self.tree = self.build(X, y)
        return self

    def _predict_one(self, x, node):
        if node["leaf"]:
            return node["prediction"]
        if x[node["feature"]] <= node["threshold"]:
            return self._predict_one(x, node["left"])
        else:
            return self._predict_one(x, node["right"])

    def predict(self, X):
        X = np.array(X)
        return np.array([self._predict_one(x, self.tree) for x in X])


In [ ]:
class ExtraTrees:
    """Extremely Randomized Trees ensemble."""

    def __init__(self, n_trees=10, max_depth=None, min_samples_split=2, sample_ratio=0.8, feature_ratio=0.8):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.sample_ratio = sample_ratio
        self.feature_ratio = feature_ratio
        self.trees = []
        self.feature_subsets = []

    def bootstrap_sample(self, X, y):
        idx = np.random.choice(len(X), int(len(X) * self.sample_ratio), replace=True)
        return X[idx], y[idx]

    def feature_subset(self, X):
        m = max(1, int(X.shape[1] * self.feature_ratio))
        return np.random.choice(X.shape[1], m, replace=False)

    def fit(self, X, y):
        self.trees = []
        self.feature_subsets = []

        for _ in range(self.n_trees):
            X_boot, y_boot = self.bootstrap_sample(X, y)
            f_idx = self.feature_subset(X_boot)

            tree = ExtraTree(max_depth=self.max_depth, min_samples_split=self.min_samples_split)
            tree.fit(X_boot[:, f_idx], y_boot)

            self.trees.append(tree)
            self.feature_subsets.append(f_idx)
        return self

    def predict(self, X):
        preds = []
        for tree, f_idx in zip(self.trees, self.feature_subsets):
            preds.append(tree.predict(X[:, f_idx]))
        preds = np.array(preds)

        final = []
        for col in preds.T:
            values, counts = np.unique(col, return_counts=True)
            final.append(values[np.argmax(counts)])
        return np.array(final)


In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

np.random.seed(42)
idx = np.random.permutation(len(X))
train_size = int(0.7 * len(X))
train_idx, test_idx = idx[:train_size], idx[train_size:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]


# Random Forest
rf = RandomForest(n_trees=10, max_depth=5)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
acc_rf = np.mean(y_pred_rf == y_test)

# Extra Trees
et = ExtraTrees(n_trees=10, max_depth=5)
et.fit(X_train, y_train)
y_pred_et = et.predict(X_test)
acc_et = np.mean(y_pred_et == y_test)


print("(RESULT) Accuracy of RandomForest:", acc_rf)
print("(RESULT) Accuracy of ExtraTrees:", acc_et)


(RESULT) Accuracy of RandomForest: 0.9555555555555556
(RESULT) Accuracy of ExtraTrees: 0.9555555555555556
